# C6 example 3/4: `KernelTensorProduct` with external circular harmonics

This same-node architecture contains exactly three 3D vectors and two scalars, packed as $3E_1\oplus5A$. Each vector $v_a$ constructs $Y(v_a)$ and carries the invariant scalar identity $q_a=a\in\{1,2,3\}$. The three filters are aggregated first, then one Wigner--Eckart tensor product is evaluated per layer. There is no additional geometry vector.

`KernelTensorProduct` expects filter features to be computed externally, so this notebook performs

$$Y_{agg}=\sum_{a=1}^{3}q_aY(v_a),\qquad m=\operatorname{KernelTP}\left(x,Y_{agg},w(I(v_1,v_2,v_3))\right).$$

The scalar identities are C6-invariant but intentionally make the aggregate sensitive to which vector is called 1, 2, or 3. The radial summary $I$ is the mean of $(\lVert(v_a)_{xy}\rVert,(v_a)_z)$ and is C6-invariant. The default circular bandlimit is $L_{full}=3$.

In [ ]:
import torch
from we3nn import CircularHarmonics, CyclicGroup, nn

torch.manual_seed(7)
torch.set_printoptions(precision=5, sci_mode=False)
G = CyclicGroup(6)
A = G.trivial_representation
E1 = G.standard_representation
regular = G.regular_representation()
input_rep = 3 * E1 + 5 * A
hidden_rep = 2 * regular
output_rep = E1 + 4 * A
harmonics = CircularHarmonics(G)  # max_frequency=None -> floor(6/2)=3
filter_rep = harmonics.rep_out
print('harmonic frequencies: 0..', harmonics.max_frequency)
print('filter representation:', filter_rep.name)
print('dimensions:', input_rep.size, 'x', filter_rep.size, '->', hidden_rep.size, '->', output_rep.size)

In [ ]:
def pack_input(vectors, scalars):
    xy = vectors[..., :, :2].reshape(*vectors.shape[:-2], 6)
    return torch.cat((xy, vectors[..., :, 2], scalars), dim=-1)

def unpack_input(x):
    xy = x[..., :6].reshape(*x.shape[:-1], 3, 2)
    return torch.cat((xy, x[..., 6:9].unsqueeze(-1)), dim=-1), x[..., 9:11]

def unpack_output(y):
    return torch.cat((y[..., :2], y[..., 2:3]), dim=-1), y[..., 3:6]

vectors = torch.tensor([[[1.0, 0.2, -0.4], [-0.3, 0.8, 1.2], [0.5, -0.7, 0.1]]])
scalars = torch.tensor([[0.6, -1.1]])
x = nn.RepresentationTensor(pack_input(vectors, scalars), input_rep)

# atan2(y, x) expresses each xy direction in the chosen coordinate basis; it is not a pairwise angle.
node_angles = torch.atan2(vectors[..., 1], vectors[..., 0])
Y = harmonics(node_angles)  # shape: [batch, 3 vector filters, harmonic components]
print('three node-vector orientations (radians):', node_angles)
print('three node-vector orientations (degrees):', torch.rad2deg(node_angles))
print('three external circular filters Y(v_a):', Y)

## Step 1: visualize the three filter-generating vectors

Every colored arrow is both a node feature and the source of one filter. The circular filter uses only the arrow's projected $xy$ direction; its planar length and $z$ component are invariant quantities handled separately by the radial network. `atan2` expresses that direction in the chosen $(x,y)$ coordinate basis. The positive $x$ axis is only the zero-angle convention, not a physically preferred direction: rotating the input shifts the angle and rotates the harmonic features according to `filter_rep`. This is not an angle between two node vectors. Each invariant identity scalar is printed below and later multiplies its harmonic filter before aggregation.

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 160, 'savefig.dpi': 240})

fig_input = plt.figure(figsize=(7.5, 6), constrained_layout=True)
ax = fig_input.add_subplot(111, projection='3d')
for index, vector in enumerate(vectors[0]):
    ax.quiver(0, 0, 0, *vector.tolist(), color=f'C{index}', linewidth=2, label=f'node feature v{index+1}')
limit = 1.15 * vectors.abs().max().item()
ax.set(xlim=(-limit, limit), ylim=(-limit, limit), zlim=(-limit, limit), xlabel='x', ylabel='y', zlabel='z', title='Each node vector constructs one shared filter')
ax.set_box_aspect((1, 1, 1)); ax.legend(fontsize=8)
print('scalar node features:', scalars[0].tolist())
print('vector identity scalars:', [1.0, 2.0, 3.0])
plt.show()

## Step 2: see the circular harmonics on their domain

A circular harmonic is a component of a representation-valued function on $S^1$. In each polar panel, the polar angle $\phi$ is a possible planar input direction and the plotted radius encodes the scalar value of one harmonic component at that direction; the radius is not the length of a physical vector. The dashed unit circle represents zero because the display uses $r(\phi)=1+0.35Y_c(\phi)/\max|Y_c|$: outside is positive and inside is negative.

Each marker is a sampled function value $Y_c(\phi_a+r\,60^\circ)$ on the C6 orbit of an actual node-vector direction $\phi_a$. Circles, squares, and triangles identify vectors 1, 2, and 3; marker color identifies the applied group rotation $r=0,\ldots,5$. The markers are not extra nodes and do not denote angles between pairs of vectors. They show how the same harmonic component changes when each vector is rotated through all six group elements.

The conversion $\phi_a=\operatorname{atan2}((v_a)_y,(v_a)_x)$ is needed because `CircularHarmonics` is parameterized by a point on $S^1$, represented by its angle. Under a C6 rotation by $\alpha$, $\phi_a\mapsto\phi_a+\alpha$, so $Y(\phi_a)$ transforms covariantly in `filter_rep`. A pairwise angle $\phi_a-\phi_b$ would instead remain invariant under a joint rotation and would not supply the nontrivial geometric filter transformation required by `KernelTensorProduct`. For a zero-length $xy$ projection the direction is undefined; a production model should mask that case even though `atan2(0,0)` numerically returns zero.

In [ ]:
import math
import matplotlib.pyplot as plt

theta_grid = torch.linspace(0.0, 2.0 * math.pi, 1441)  # 0.25-degree sampling
Y_circle = harmonics(theta_grid).detach()
orbit_angles = node_angles[0, :, None] + torch.arange(6)[None, :] * (2.0 * math.pi / 6.0)
Y_orbit = harmonics(orbit_angles).detach()

harmonic_labels = []
for frequency, mode in harmonics._layout:
    if mode == 'pair':
        harmonic_labels.extend((f'cos({frequency}θ)', f'sin({frequency}θ)'))
    elif mode == 'conjugate_pair':
        harmonic_labels.extend((f'cos({frequency}θ)', f'-sin({frequency}θ)'))
    else:
        harmonic_labels.append(f'{mode}({frequency}θ)')

fig_circle, axes = plt.subplots(2, 4, figsize=(18, 10), dpi=180, subplot_kw={'projection': 'polar'}, constrained_layout=True)
for component, (ax, label) in enumerate(zip(axes.flat, harmonic_labels)):
    scale = Y_circle[:, component].abs().max().clamp_min(1e-8)
    radius = 1.0 + 0.35 * Y_circle[:, component] / scale
    orbit_radius = 1.0 + 0.35 * Y_orbit[..., component] / scale
    ax.plot(theta_grid, torch.ones_like(theta_grid), color='gray', linestyle='--', linewidth=0.8)
    ax.plot(theta_grid, radius, linewidth=2.4)
    for vector_index, marker in enumerate(('o', 's', '^')):
        ax.scatter(orbit_angles[vector_index], orbit_radius[vector_index], c=torch.arange(6), cmap='hsv', marker=marker, s=48, edgecolors='black', linewidths=0.35, zorder=3)
    ax.set_ylim(0.6, 1.4); ax.set_yticklabels([]); ax.set_title(label)
axes.flat[-1].set_axis_off()
fig_circle.suptitle('Every circular-harmonic component used by C6 (frequencies 0, 1, 2, 3)', fontsize=14)
plt.show()

## Step 3: build the externally filtered architecture

For each layer, the three identity-weighted harmonics are first added inside their common representation space. The finite-group tensors $C_p$ then couple `x` to that single aggregate using one coefficient vector $w_p(I)$:

$$z_o=\sum_p w_p(I)(C_p)_{oij}x_i\left[\sum_{a=1}^{3}q_aY_j(v_a)\right].$$

The hidden regular representations admit coordinatewise `PointActiv`. Each layer now executes only one tensor product; the input and output layers still have separate parameters.

In [ ]:
class ExternalHarmonicNetwork(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.harmonics = harmonics
        self.input_layer = nn.KernelTensorProduct(
            input_rep, filter_rep, hidden_rep, shared_weights=False
        )
        self.activation = nn.PointActiv(hidden_rep, torch.relu)
        self.output_layer = nn.KernelTensorProduct(
            hidden_rep, filter_rep, output_rep, shared_weights=False
        )
        self.register_buffer('vector_ids', torch.tensor([1.0, 2.0, 3.0]), persistent=False)
        # Each radial network runs once per node, not once per vector.
        self.radial_in = torch.nn.Sequential(
            torch.nn.Linear(2, 16), torch.nn.SiLU(),
            torch.nn.Linear(16, self.input_layer.weight_numel),
        )
        self.radial_out = torch.nn.Sequential(
            torch.nn.Linear(2, 16), torch.nn.SiLU(),
            torch.nn.Linear(16, self.output_layer.weight_numel),
        )

    def forward(self, features):
        # Recover the same three physical vectors already stored inside x.
        node_vectors, _ = unpack_input(features.tensor)
        # Orientations in the chosen xy basis: a C6 rotation shifts every angle by the group angle.
        # Pairwise relative angles would stay invariant and would not form the required filter rep.
        node_angles = torch.atan2(node_vectors[..., 1], node_vectors[..., 0])
        individual_values = self.harmonics(node_angles)
        individual_filters = nn.RepresentationTensor(individual_values, filter_rep)
        identities = self.vector_ids.to(device=individual_values.device, dtype=individual_values.dtype)
        aggregated_values = (individual_values * identities.view(1, 3, 1)).sum(dim=-2)
        aggregated_filter = nn.RepresentationTensor(aggregated_values, filter_rep)
        per_vector_invariants = torch.stack(
            (torch.linalg.vector_norm(node_vectors[..., :2], dim=-1), node_vectors[..., 2]),
            dim=-1,
        )
        invariant_summary = per_vector_invariants.mean(dim=-2)
        w_in = self.radial_in(invariant_summary)
        w_out = self.radial_out(invariant_summary)

        # One aggregated filter -> one tensor product per layer.
        h_pre = self.input_layer(features, aggregated_filter, w_in)
        h = self.activation(h_pre)
        y = self.output_layer(h, aggregated_filter, w_out)
        return y, individual_filters, aggregated_filter, w_in, w_out, h_pre, h

model = ExternalHarmonicNetwork().eval()
y, Y_individual, Y_aggregated, w_in, w_out, h_pre, h = model(x)
print('input/output reduced-weight counts:', model.input_layer.weight_numel, model.output_layer.weight_numel)
print('vector identity scalars:', model.vector_ids.tolist())
print('one input-layer reduced-weight vector:', w_in[0])
print('aggregated filter sum_a identity[a] * Y(v_a):', Y_aggregated.tensor)
print('hidden before PointActiv:', h_pre.tensor)
print('hidden after  PointActiv:', h.tensor)
print('physical output (vector, scalars):', unpack_output(y.tensor))
kernel_basis = model.input_layer.sample_kernel_basis(Y_aggregated)
print('single sampled basis shape [batch, paths, out, in]:', tuple(kernel_basis.shape))

## Step 4: rotate the node and all three derived filters together

Rotating `x` automatically rotates the three vectors stored inside it. Their individual filters and identity-weighted aggregate transform covariantly, while the symmetric radial summary and reduced-weight vector stay fixed.

In [ ]:
errors = []
for k, element in enumerate(G.elements):
    x_k = x.transform_fibers(element)
    y_k, Y_individual_k, Y_aggregated_k, w_in_k, w_out_k, _, _ = model(x_k)
    expected_k = y.transform_fibers(element)
    error = (y_k.tensor - expected_k.tensor).abs().max().item()
    errors.append(error)
    torch.testing.assert_close(y_k.tensor, expected_k.tensor, atol=5e-5, rtol=5e-5)
    torch.testing.assert_close(w_in_k, w_in, atol=1e-6, rtol=1e-6)
    torch.testing.assert_close(w_out_k, w_out, atol=1e-6, rtol=1e-6)
    in_vectors_k, in_scalars_k = unpack_input(x_k.tensor)
    out_vector_k, out_scalars_k = unpack_output(y_k.tensor)
    print(f'rotation {k}: angle={60*k:3d} degrees')
    print('  three circular filters:', Y_individual_k.tensor[0].tolist())
    print('  identity-weighted aggregate:', Y_aggregated_k.tensor[0].tolist())
    print('  input vectors   :', in_vectors_k[0].tolist())
    print('  input scalars   :', in_scalars_k[0].tolist())
    print('  output vector   :', out_vector_k[0].tolist())
    print('  output scalars  :', out_scalars_k[0].tolist())
    print(f'  max equivariance error: {error:.3e}')

print('maximum over all rotations:', max(errors))

## Step 5: follow the harmonic filter into the regular hidden state

The left heatmap shows $Y(v_1)$, $Y(v_2)$, $Y(v_3)$, and the resulting $Y_{agg}=Y(v_1)+2Y(v_2)+3Y(v_3)$ as a fourth block. The right panel is the activated output of the single tensor product.

In [ ]:
import matplotlib.pyplot as plt

hidden_by_rotation, harmonic_by_rotation, aggregate_by_rotation = [], [], []
output_vectors, output_scalars = [], []
for element in G.elements:
    x_k = x.transform_fibers(element)
    y_k, Y_individual_k, Y_aggregated_k, _, _, _, h_k = model(x_k)
    vector_k, scalars_k = unpack_output(y_k.tensor)
    hidden_by_rotation.append(h_k.tensor[0].detach())
    harmonic_by_rotation.append(Y_individual_k.tensor[0].detach())
    aggregate_by_rotation.append(Y_aggregated_k.tensor[0].detach())
    output_vectors.append(vector_k[0].detach())
    output_scalars.append(scalars_k[0].detach())
hidden_by_rotation = torch.stack(hidden_by_rotation).cpu()
harmonic_by_rotation = torch.stack(harmonic_by_rotation).cpu()
aggregate_by_rotation = torch.stack(aggregate_by_rotation).cpu()
output_vectors = torch.stack(output_vectors).cpu()
output_scalars = torch.stack(output_scalars).cpu()
angles_deg = torch.arange(6) * 60
colors = plt.cm.hsv(torch.linspace(0, 5/6, 6).numpy())

harmonic_labels = []
for frequency, mode in harmonics._layout:
    if mode == 'pair':
        harmonic_labels.extend((f'cos({frequency}θ)', f'sin({frequency}θ)'))
    elif mode == 'conjugate_pair':
        harmonic_labels.extend((f'cos({frequency}θ)', f'-sin({frequency}θ)'))
    else:
        harmonic_labels.append(f'{mode}({frequency}θ)')

all_filter_blocks = torch.cat((harmonic_by_rotation, aggregate_by_rotation.unsqueeze(-2)), dim=-2)
flat_harmonics = all_filter_blocks.reshape(6, -1)
flat_labels = [f'{name}:{label}' for name in ('v1', 'v2', 'v3', 'agg') for label in harmonic_labels]
fig_state, (ax_harm, ax_hidden) = plt.subplots(1, 2, figsize=(20, 6), dpi=170, constrained_layout=True)
harmonic_image = ax_harm.imshow(flat_harmonics, aspect='auto', cmap='coolwarm')
for boundary in (6.5, 13.5, 20.5):
    ax_harm.axvline(boundary, color='white', linewidth=2)
ax_harm.set(xticks=range(28), xticklabels=flat_labels, yticks=range(6), yticklabels=[f'{a}°' for a in angles_deg.tolist()], xlabel='individual filters and identity-weighted aggregate', ylabel='C6 rotation', title='Y(v1) | Y(v2) | Y(v3) | Y_agg')
ax_harm.tick_params(axis='x', labelrotation=90, labelsize=6)
fig_state.colorbar(harmonic_image, ax=ax_harm, shrink=0.75)

image = ax_hidden.imshow(hidden_by_rotation, aspect='auto', cmap='coolwarm')
ax_hidden.axvline(5.5, color='white', linewidth=2)
ax_hidden.set(xticks=range(12), yticks=range(6), yticklabels=[f'{a}°' for a in angles_deg.tolist()], xlabel='regular component (copies 1 | 2)', ylabel='rotation', title='Hidden 2 Reg(C6) after PointActiv')
fig_state.colorbar(image, ax=ax_hidden, shrink=0.75)
plt.show()

## Step 6: inspect the sampled kernel itself

Because the aggregate is formed before contraction, `sample_kernel_basis` is called once on $Y_{agg}$. It returns $K_p(Y_{agg})$, the one reduced-weight vector contracts its path axis, and the assertion verifies $h_{pre}=K(Y_{agg})x$.

In [ ]:
basis_for_aggregate = model.input_layer.sample_kernel_basis(Y_aggregated)[0].detach()
effective_kernel = torch.einsum('p,poi->oi', w_in[0].detach(), basis_for_aggregate).cpu()
torch.testing.assert_close(h_pre.tensor[0], effective_kernel @ x.tensor[0], atol=2e-5, rtol=2e-5)

fig_kernel, ax_kernel = plt.subplots(figsize=(8.5, 6), dpi=170, constrained_layout=True)
kernel_image = ax_kernel.imshow(effective_kernel, aspect='auto', cmap='coolwarm')
ax_kernel.set(xlabel='input coordinate', ylabel='hidden coordinate', title='One kernel from the aggregated filter')
fig_kernel.colorbar(kernel_image, ax=ax_kernel, shrink=0.75)
plt.show()

## Step 7: unpack the final tensor-product output

The final layer produces one $xy$ vector irrep and four trivial coordinates. We combine the first trivial coordinate with $xy$ to reconstruct the physical 3D vector; the remaining three are displayed as scalar channels.

In [ ]:
fig_output = plt.figure(figsize=(14, 6), dpi=170, constrained_layout=True)
ax_vec = fig_output.add_subplot(1, 2, 1, projection='3d')
for k, (vector, color) in enumerate(zip(output_vectors, colors)):
    ax_vec.quiver(0, 0, 0, *vector.tolist(), color=color, linewidth=2, label=f'{60*k}°')
ax_vec.set(xlabel='x', ylabel='y', zlabel='z', title='KernelTensorProduct output vector')
output_limit = max(1e-3, 1.15 * output_vectors.abs().max().item())
ax_vec.set_xlim(-output_limit, output_limit); ax_vec.set_ylim(-output_limit, output_limit); ax_vec.set_zlim(-output_limit, output_limit); ax_vec.set_box_aspect((1, 1, 1))
ax_vec.legend(ncols=2, fontsize=7)

ax_scalar = fig_output.add_subplot(1, 2, 2)
for channel in range(3):
    ax_scalar.plot(angles_deg, output_scalars[:, channel], marker='o', label=f'output scalar {channel+1}')
ax_scalar.set(xticks=angles_deg.tolist(), xlabel='C6 rotation', ylabel='value', title='KernelTensorProduct output scalars')
ax_scalar.grid(alpha=0.3); ax_scalar.legend()
plt.show()